# From dense to hybrid to reranked — every step measured

`notebooks/use_cases_demo.ipynb` found the gaps. This notebook closes them one at a time and
measures each with `evaluate()`: **hit@1** (right chunk first), **hit@5**, **MRR** (mean 1/rank).

1. Hybrid retrieval: dense ∪ BM25 keyword
2. Reranking with a cross-encoder
3. Grounded answers: query rewrite, refusal, validated citations
4. Languages, fixed: a multilingual embedder
5. Structure: graph edges, LLM-extracted entities, `related()`
6. Conversation memory: sessions
7. Speed: is any of this too slow for Python?

Needs the **Python 3 (trading)** kernel; Ollama with `nomic-embed-text`, `bge-m3`, `qwen2.5:7b-instruct`;
`pip install slim-llm-memory[rerank,graph]`.

In [1]:
import sys, time
from pathlib import Path
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))

from slim_llm_memory import library, evaluate, Reranker

db = library(ROOT / ".accuracy_nb")
docs = db.topic("slim-llm-memory")
docs.add([ROOT / "README.md", ROOT / "docs" / "IMPLEMENTATION.md"])
LLM = "qwen2.5:7b-instruct"

# Eight questions with checkable answers. The first four carry the product name — the
# case that broke dense retrieval in the previous notebook.
CASES = [
    ("In slim-llm-memory, which file is the atomic commit point of a flush?",          "manifest"),
    ("In slim-llm-memory, at what tombstone ratio does compaction happen?",             "20%"),
    ("Which local embedding model does slim-llm-memory use by default?",                "nomic"),
    ("How does slim-llm-memory stop two processes from writing the same index?",        "lock"),
    ("Which file is the atomic commit point when flushing the index to disk?",           "manifest"),
    ("When are tombstoned items compacted away — at what percentage?",                   "20%"),
    ("Which Ollama embedding model is the default?",                                     "nomic"),
    ("What prevents a second writer process from opening the same index directory?",    "lock"),
]
docs

topic('slim-llm-memory': 2 doc(s), 76 chunks, ollama:nomic-embed-text, /home/trbck/workspace/slim-llm-memory/.accuracy_nb/slim-llm-memory)

## 1. Hybrid retrieval

`ask(mode=...)`: **dense** is cosine only, **keyword** is BM25 only, **hybrid** (the new default) fuses both
by min-max-normalised score. A rare token like *manifest.json* or *20%* is exactly what BM25 is good at.

In [2]:
for mode in ("dense", "keyword", "hybrid"):
    print(evaluate(docs, CASES, k=5, mode=mode, min_score=0.0, label=mode), "\n")

evaluate(dense, 8 cases, k=5):  hit@1 0.38 · hit@5 0.62 · MRR 0.47
     —  In slim-llm-memory, which file is the atomic commit point of  expects 'manifest'
     —  In slim-llm-memory, at what tombstone ratio does compaction   expects '20%'
     4  Which local embedding model does slim-llm-memory use by defa  expects 'nomic'
     —  How does slim-llm-memory stop two processes from writing the  expects 'lock'
     2  Which file is the atomic commit point when flushing the inde  expects 'manifest'
     1  When are tombstoned items compacted away — at what percentag  expects '20%'
     1  Which Ollama embedding model is the default?                  expects 'nomic'
     1  What prevents a second writer process from opening the same   expects 'lock' 



evaluate(keyword, 8 cases, k=5):  hit@1 0.25 · hit@5 0.88 · MRR 0.50
     2  In slim-llm-memory, which file is the atomic commit point of  expects 'manifest'
     2  In slim-llm-memory, at what tombstone ratio does compaction   expects '20%'
     5  Which local embedding model does slim-llm-memory use by defa  expects 'nomic'
     —  How does slim-llm-memory stop two processes from writing the  expects 'lock'
     2  Which file is the atomic commit point when flushing the inde  expects 'manifest'
     1  When are tombstoned items compacted away — at what percentag  expects '20%'
     3  Which Ollama embedding model is the default?                  expects 'nomic'
     1  What prevents a second writer process from opening the same   expects 'lock' 



evaluate(hybrid, 8 cases, k=5):  hit@1 0.38 · hit@5 0.88 · MRR 0.56
     2  In slim-llm-memory, which file is the atomic commit point of  expects 'manifest'
     4  In slim-llm-memory, at what tombstone ratio does compaction   expects '20%'
     5  Which local embedding model does slim-llm-memory use by defa  expects 'nomic'
     —  How does slim-llm-memory stop two processes from writing the  expects 'lock'
     2  Which file is the atomic commit point when flushing the inde  expects 'manifest'
     1  When are tombstoned items compacted away — at what percentag  expects '20%'
     1  Which Ollama embedding model is the default?                  expects 'nomic'
     1  What prevents a second writer process from opening the same   expects 'lock' 



Where each hit came from is visible on the hit itself (`meta["via"]`):

In [3]:
docs.ask("In slim-llm-memory, which file is the atomic commit point of a flush?", k=3, min_score=0.0)

ask('In slim-llm-memory, which file is the atomic commit point of a flush?')  3 hit(s) · hybrid · embed 84 ms · scan 0.40 ms
   1  0.72  IMPLEMENTATION.md#19     # 8. flush to disk (atomic — safe to crash mid-anything) mem.f  [both]
   2  0.61  README.md#10             ## Persistence model Files in your index directory: ``` items.  [both]
   3  0.67  IMPLEMENTATION.md#51     ## 14. First commit checklist (for a fresh project) - [ ] `pyp  [both]

## 2. Reranking

`ask(rerank=True)` retrieves the top 4·k candidates and lets a cross-encoder (`bge-reranker-v2-m3`) read
each (question, chunk) pair. It is the most accurate thing here and by far the slowest, so it is opt-in.

In [4]:
rr = Reranker.cross_encoder()
t0 = time.perf_counter(); rr.score("warm-up", ["warm-up"]); print(f"model load: {time.perf_counter()-t0:.1f} s")
print(evaluate(docs, CASES, k=5, min_score=0.0, rerank=rr, label="hybrid + rerank"))
r = docs.ask("How does slim-llm-memory stop two processes from writing the same index?", k=3, min_score=0.0, rerank=rr)
print(f"\nreranking 20 candidates took {r.rerank_ms:.0f} ms")
r

Loading weights:   0%|          | 0/393 [00:00<?, ?it/s]

model load: 8.8 s


evaluate(hybrid + rerank, 8 cases, k=5):  hit@1 0.62 · hit@5 1.00 · MRR 0.76
     1  In slim-llm-memory, which file is the atomic commit point of  expects 'manifest'
     1  In slim-llm-memory, at what tombstone ratio does compaction   expects '20%'
     3  Which local embedding model does slim-llm-memory use by defa  expects 'nomic'
     4  How does slim-llm-memory stop two processes from writing the  expects 'lock'
     1  Which file is the atomic commit point when flushing the inde  expects 'manifest'
     1  When are tombstoned items compacted away — at what percentag  expects '20%'
     1  Which Ollama embedding model is the default?                  expects 'nomic'
     2  What prevents a second writer process from opening the same   expects 'lock'



reranking 20 candidates took 22520 ms


ask('How does slim-llm-memory stop two processes from writing the same index?')  3 hit(s) · hybrid · embed 706 ms · scan 0.45 ms · reranked (cross-encoder:BAAI/bge-reranker-v2-m3) in 22520 ms
   1  0.70  IMPLEMENTATION.md#0      # slim-llm-memory — Implementation Plan > **Goal.** A drop-in   [both]
   2  0.54  README.md#22             ## Migration paths When the slim stack stops being enough, swa  [keyword]
   3  0.60  README.md#13             ## Topic store: fast context for an LLM working on one topic …  [both]

## 3. Grounded answers, hardened

`answer()` now returns an `Answer`: a string that also carries the hits it used, the context, and the
citations it actually made (dangling `[n]` markers are removed). `rewrite=True` first turns the question
into a search query with one short model call; `refuse_below` refuses without calling the model when
nothing relevant was retrieved.

In [5]:
a = docs.answer("In slim-llm-memory, at what tombstone ratio does compaction happen?", model=LLM, rewrite=True)
print("query used :", a.query)
print("citations  :", a.citations)
print("answer     :", a)

query used : slim-llm-memory tombstone ratio compaction
citations  : [3]
answer     : Compaction happens when more than 20% of the items in `items.jsonl` are tombstoned [3].


In [6]:
a = docs.answer("what is the capital of France?", model=LLM, refuse_below=0.45)
print("refused:", a.refused, "→", a)

refused: True → I don't have anything about that in this store.


## 4. Languages, fixed

Same German notes as before, now embedded with `bge-m3` (multilingual). The English questions that missed
with `nomic-embed-text` land.

In [7]:
ml = library(ROOT / ".accuracy_nb_ml", embedder="ollama:bge-m3")
de = ml.topic("privat")
de.add({
    "einkauf.md":  "Einkaufsliste: Milch, Brot, Eier, Butter und Kaffee.",
    "zahnarzt.md": "Zahnarzttermin am Dienstag um 9 Uhr, bitte nicht vergessen.",
    "steuer.md":   "Die Steuererklärung muss bis Ende Juli eingereicht werden.",
})
print(evaluate(de, [("when is the dentist appointment?", "zahnarzt.md"),
                    ("what do I need from the supermarket?", "einkauf.md"),
                    ("tax filing deadline", "steuer.md"),
                    ("Wann ist der Zahnarzt?", "zahnarzt.md")], k=3, mode="dense", min_score=0.0, label="bge-m3, EN→DE"))
ml.close()

evaluate(bge-m3, EN→DE, 4 cases, k=3):  hit@1 1.00 · hit@3 1.00 · MRR 1.00
     1  when is the dentist appointment?      expects 'zahnarzt.md'
     1  what do I need from the supermarket?  expects 'einkauf.md'
     1  tax filing deadline                   expects 'steuer.md'
     1  Wann ist der Zahnarzt?                expects 'zahnarzt.md'


## 5. Structure: a graph next to the vectors

`[[wikilinks]]` in a note become typed edges automatically. `link()` adds your own. `related()` blends
cosine similarity (0.6) with graph edges (0.4). And `add(enrich=model)` lets a local model extract entities
and relations from each new chunk — slow (one call per chunk), so it is opt-in — after which
`entities()`, `ask(entity=...)` and `neighbours("Postgres")` answer *structural* questions.

In [8]:
ops = db.topic("ops notes")
t0 = time.perf_counter()
ops.add({
    "postgres.md": "Postgres connection pool exhausted last night. We added pgbouncer in front of it; see [[runbook]].",
    "runbook.md":  "Runbook: restart the API pods after any Postgres failover. Grafana dashboard 'db-health' shows pool usage.",
    "cdn.md":      "Egress cost is driven by image traffic; moving static assets to Cloudflare CDN cuts it by roughly a third.",
}, enrich=LLM)
print(f"enrichment: {time.perf_counter()-t0:.0f} s for 3 chunks")
print("entities:", ops.entities())
print("edges:", ops.graph.edges())

enrichment: 133 s for 3 chunks
entities: {'Postgres': 2, 'API pods': 1, 'Cloudflare CDN': 1, 'db-health': 1, 'Egress cost': 1, 'Grafana dashboard': 1, 'image traffic': 1, 'pgbouncer': 1, 'pool usage': 1, 'restart': 1, 'runbook': 1, 'static assets': 1}
edges: [('postgres.md', 'Postgres', 'mentions', 1.0), ('postgres.md', 'pgbouncer', 'mentions', 1.0), ('postgres.md', 'runbook', 'mentions', 1.0), ('postgres.md', 'runbook.md', 'links', 1.0), ('pgbouncer', 'runbook', 'see', 1.0), ('we', 'pgbouncer', 'added', 1.0), ('runbook.md', 'restart', 'mentions', 1.0), ('runbook.md', 'API pods', 'mentions', 1.0), ('runbook.md', 'Postgres', 'mentions', 1.0), ('runbook.md', 'Grafana dashboard', 'mentions', 1.0), ('runbook.md', 'db-health', 'mentions', 1.0), ('runbook.md', 'pool usage', 'mentions', 1.0), ('restart', 'Postgres failover', 'depends_on', 1.0), ('Grafana dashboard db-health', 'pool usage', 'shows', 1.0), ('cdn.md', 'Egress cost', 'mentions', 1.0), ('cdn.md', 'image traffic', 'mentions', 1.0),

In [9]:
print("neighbours('Postgres'):", ops.neighbours("Postgres"))
ops.ask("what happened last night?", k=3, min_score=0.0, entity="Postgres")

neighbours('Postgres'): [('postgres.md', 'mentions', 1.0), ('runbook.md', 'mentions', 1.0)]


ask('what happened last night?')  2 hit(s) · hybrid · embed 899 ms · scan 1.11 ms
   1  0.44  postgres.md#0            Postgres connection pool exhausted last night. We added pgboun  [both]
   2  0.41  runbook.md#0             Runbook: restart the API pods after any Postgres failover. Gra  [dense]

In [10]:
ops.related("postgres.md", k=3)

ask("related('postgres.md')")  2 hit(s) · related · embed 0 ms · scan 0.30 ms
   1  0.72  runbook.md#0             Runbook: restart the API pods after any Postgres failover. Gra  [graph]
   2  0.44  cdn.md#0                 Egress cost is driven by image traffic; moving static assets t  [vector]

## 6. Conversation memory

A session is a topic whose docs are turns. Later prompts, differently worded, get the turns back; `summary()`
is one model call over the transcript.

In [11]:
s = db.session("2026-09-04 pairing")
s.turn("user", "The flaky test was caused by a shared tmp dir between tests.")
s.turn("assistant", "Fixed: every test now uses its own tmp_path fixture.")
s.turn("user", "Let's keep the numpy scan until the index passes 100k chunks, then look at faiss.")
print(s)
print(s.recall("why were the tests unreliable?", k=1, min_score=0.0).top.text)
print(s.recall("when do we switch to an ANN index?", k=1, min_score=0.0).top.text)
print()
print(s.summary(model=LLM))

session('2026-09-04 pairing': 3 turns, /home/trbck/workspace/slim-llm-memory/.accuracy_nb/_sessions/2026-09-04-pairing)
The flaky test was caused by a shared tmp dir between tests.
Let's keep the numpy scan until the index passes 100k chunks, then look at faiss.



- Flaky test resolved by using individual `tmp_path` for each test.
- Decision to maintain numpy scan until the index processes over 100k chunks.
- Next step is to evaluate Faiss after the numpy scan threshold is met.
- No open questions noted in this conversation.
- Findings indicate that shared tmp dir was the root cause of flakiness.


## 7. Speed: does any of this need Cython or Go?

Budget from the roadmap: hybrid query < 20 ms at 50k chunks; BM25 build < 5 s at 50k or cached on disk.
Synthetic 50k chunks of 100 words, offline embedder, so these are the library's own costs.

In [12]:
import numpy as np, tempfile
from slim_llm_memory import topic
from slim_llm_memory.keyword import BM25
rng = np.random.default_rng(0)
vocab = [f"w{i}" for i in range(20_000)]
texts = [" ".join(rng.choice(vocab, 100)) for _ in range(50_000)]

t0 = time.perf_counter(); ix = BM25(texts); build = time.perf_counter() - t0
t0 = time.perf_counter()
for i in range(50): ix.search(" ".join(rng.choice(vocab, 6)), k=10)
q_bm25 = (time.perf_counter() - t0) / 50 * 1000
print(f"BM25 build 50k chunks: {build:.1f} s   (cached on disk per store version → paid once per change)")
print(f"BM25 query:            {q_bm25:.2f} ms")

with tempfile.TemporaryDirectory() as tmp:
    big = topic("big", path=tmp, embedder="noop:768", overlap=0)
    t0 = time.perf_counter(); big.add({f"d{i}.md": t for i, t in enumerate(texts[:20_000])}); add_s = time.perf_counter() - t0
    lat = []
    for i in range(30):
        q = " ".join(rng.choice(vocab, 6)); t0 = time.perf_counter(); big.ask(q, k=5, min_score=0.0); lat.append((time.perf_counter()-t0)*1000)
    print(f"topic.add 20k chunks (noop embedder): {add_s:.1f} s")
    print(f"hybrid ask at 20k chunks: p50 {np.median(lat):.1f} ms, p95 {np.percentile(lat, 95):.1f} ms")
    big.close()

BM25 build 50k chunks: 5.6 s   (cached on disk per store version → paid once per change)
BM25 query:            2.74 ms


topic.add 20k chunks (noop embedder): 5.2 s
hybrid ask at 20k chunks: p50 4.6 ms, p95 6.0 ms


**Verdict.** The hybrid query is inside budget in plain Python + numpy; the BM25 build is the only
multi-second step and it is paid once per store change and cached in `bm25.npz`. Nothing here needs
Cython or Go today. The first thing that would: BM25 tokenising at 500k+ chunks (a Go or Cython tokeniser
would cut the build ~10×), and the cross-encoder, which is bound by the model, not the language.

## What is still missing

- **Evaluation at scale**: `evaluate()` needs a real question set per topic; the eight questions here are a smoke test.
- **Reranker speed on CPU**: ~250 ms per candidate on a loaded box. A GPU, or a smaller reranker, or reranking only when the top two scores are close.
- **Ontology proper**: entities and relations exist now, but there is no schema, no inference, and extraction quality is whatever the local model gives you. A curated entity list to normalise against would help most.
- **Hybrid tuning**: `FUSION_ALPHA = 0.5` was chosen on eight questions. Re-tune per corpus with `evaluate()`.

In [13]:
db.close()